In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# อ่านข้อมูล
df = pd.read_excel("/content/new data.xlsx", sheet_name="Sheet2")  # เปลี่ยนชื่อไฟล์ตามจริง

In [ ]:
# ลบคอลัมน์ Date หากมี (เพราะไม่สามารถใช้กับโมเดลโดยตรงได้)
df = df.drop(columns=["Date"], errors='ignore')

In [ ]:
# แยก features และ target
# Drop rows where the target variable 'dBZ' is missing
df_cleaned = df.dropna(subset=["dBZ"])
X = df_cleaned.drop(columns=["dBZ"])
y = df_cleaned["dBZ"]

In [ ]:
# ฟังก์ชันวัด MAE RMSE
def evaluate_model(X_imputed, y):
    X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse) # Calculate RMSE from MSE
    r2 = model.score(X_test_scaled, y_test) # Calculate R2 score
    return mae, rmse, r2

In [ ]:
# ----------- วิธีที่ 1: เติมค่าด้วย KNN -----------
# Replace '-' with NaN
X = X.replace('-', np.nan)
knn_imputer = KNNImputer(n_neighbors=5)
X_knn = knn_imputer.fit_transform(X)
mae_knn, rmse_knn, r2_knn = evaluate_model(X_knn, y)

In [ ]:
# ----------- วิธีที่ 2: เติมค่าด้วย Random Forest -----------
rf_imputer = IterativeImputer(estimator=RandomForestRegressor(random_state=42), random_state=42)
X_rf = rf_imputer.fit_transform(X)
mae_rf, rmse_rf, r2_rf = evaluate_model(X_rf, y)

In [ ]:
# --- แสดงผลลัพธ์ ---
print(f"KNN Imputer:     MAE = {mae_knn:.2f}, RMSE = {rmse_knn:.2f}, R2 = {r2_knn:.2f}")
print(f"RF Imputer:      MAE = {mae_rf:.2f}, RMSE = {rmse_rf:.2f}, R2 = {r2_rf:.2f}")

KNN Imputer:     MAE = 16.05, RMSE = 18.69, R2 = 0.03
RF Imputer:      MAE = 15.80, RMSE = 18.87, R2 = 0.01


In [ ]:
# ตรวจสอบตัวอย่าง
print("KNN Imputed Data")
print(pd.DataFrame(X_knn, columns=X.columns).head())
print("\nRF Imputed Data")
print(pd.DataFrame(X_rf, columns=X.columns).head())

KNN Imputed Data
   Showalter  Lifted   SWEAT     K  Cross  Vertical  TT Totals   CAPE    CIN  \
0      13.29   16.28  244.92 -27.1    5.7      19.7       25.4   0.00   0.00   
1      10.78   11.77  127.18 -22.1   10.3      19.3       29.6   0.00   0.00   
2       3.53    4.54  362.30 -14.3   19.3      19.7       39.0  15.68 -46.73   
3       4.66    5.24  200.40 -18.0   18.8      19.3       38.1   0.00 -50.25   
4       7.09    8.06  170.99  -0.7   15.3      18.3       33.6   0.00   0.00   

    BRN  LCL Temp   LCL P  LCL P.1   MML θ  MML q  Thickness   PWAT  
0  0.00    276.14  783.20   314.16  296.13   6.16     5749.0  12.64  
1  0.00    281.45  839.90   319.96  295.87   8.30     5746.0  16.33  
2  0.86    288.03  903.90   331.01  296.49  11.91     5729.0  25.68  
3  0.00    285.99  879.04   327.88  296.74  10.71     5734.0  22.45  
4  0.00    285.51  868.80   327.84  297.24  10.51     5733.0  22.85  

RF Imputed Data
   Showalter  Lifted   SWEAT     K  Cross  Vertical  TT Totals   

In [ ]:
# บันทึก KNN Imputed
pd.DataFrame(X_knn, columns=X.columns).to_excel("data_knn_imputed.xlsx", index=False)

# บันทึก RF Imputed
pd.DataFrame(X_rf, columns=X.columns).to_excel("data_rf_imputed.xlsx", index=False)

print("บันทึกไฟล์ Excel เสร็จเรียบร้อย ✅")

บันทึกไฟล์ Excel เสร็จเรียบร้อย ✅


mean

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

In [ ]:
# อ่านข้อมูล
df = pd.read_excel("/content/new data.xlsx", sheet_name="Sheet2")  # เปลี่ยนชื่อไฟล์ตามจริง

In [ ]:
print("ก่อนเติมค่า:")
print(df)

ก่อนเติมค่า:
          Date Showalter Lifted   SWEAT     K Cross Vertical TT Totals  \
0   2018-02-01     13.29  16.28  244.92 -27.1   5.7     19.7      25.4   
1   2018-02-02     10.78  11.77  127.18 -22.1  10.3     19.3      29.6   
2   2018-02-03      3.53   4.54   362.3 -14.3  19.3     19.7        39   
3   2018-02-04      4.66   5.24   200.4   -18  18.8     19.3      38.1   
4   2018-02-05      7.09   8.06  170.99  -0.7  15.3     18.3      33.6   
..         ...       ...    ...     ...   ...   ...      ...       ...   
367 2021-02-08     -4.55  -5.04  352.01    40  25.5     30.3      55.8   
368 2021-02-09      0.55   2.57  162.58     2    23     23.7      46.7   
369 2021-02-10      8.44   9.02  142.21  -4.7  13.7     19.7      33.4   
370 2021-02-11     10.57  10.84  103.01 -18.9  10.1     20.1      30.2   
371 2021-02-12      5.79   7.91  123.39 -11.9  13.9     24.9      38.8   

       CAPE     CIN   BRN LCL Temp   LCL P LCL P.1   MML θ  MML q Thickness  \
0         0       0

In [ ]:
# ลบคอลัมน์ Date หากมี (เพราะไม่สามารถใช้กับโมเดลโดยตรงได้)
df = df.drop(columns=["Date"], errors='ignore')

In [ ]:
# สร้าง Imputer แบบ mean
#imputer = SimpleImputer(strategy='mean')
imputer = KNNImputer(n_neighbors=2)
df_filled = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

In [ ]:
# แปลง '-' ให้เป็น NaN
df = df.replace("-", np.nan)

In [ ]:
# 2) เลือกเฉพาะคอลัมน์ตัวเลข
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")  # แปลงคอลัมน์ตัวเลข

numeric_cols = df.select_dtypes(include=[np.number]).columns

/tmp/ipython-input-69854565.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")  # แปลงคอลัมน์ตัวเลข


In [ ]:
# 3) เติมค่า missing ด้วย mean
#imputer = SimpleImputer(strategy="mean")
#df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print("\nหลังเติมค่าเฉลี่ย:")
print(df)


หลังเติมค่าเฉลี่ย:
     Showalter  Lifted   SWEAT     K  Cross  Vertical  TT Totals    CAPE  \
0        13.29   16.28  244.92 -27.1    5.7      19.7       25.4    0.00   
1        10.78   11.77  127.18 -22.1   10.3      19.3       29.6    0.00   
2         3.53    4.54  362.30 -14.3   19.3      19.7       39.0   15.68   
3         4.66    5.24  200.40 -18.0   18.8      19.3       38.1    0.00   
4         7.09    8.06  170.99  -0.7   15.3      18.3       33.6    0.00   
..         ...     ...     ...   ...    ...       ...        ...     ...   
367      -4.55   -5.04  352.01  40.0   25.5      30.3       55.8  318.19   
368       0.55    2.57  162.58   2.0   23.0      23.7       46.7    6.52   
369       8.44    9.02  142.21  -4.7   13.7      19.7       33.4    0.00   
370      10.57   10.84  103.01 -18.9   10.1      20.1       30.2    0.00   
371       5.79    7.91  123.39 -11.9   13.9      24.9       38.8    0.00   

        CIN   BRN  LCL Temp   LCL P  LCL P.1   MML θ  MML q  Thickn

In [ ]:
# แยก features และ target
# Drop rows where the target variable 'dBZ' is missing
df_cleaned = df.dropna(subset=["dBZ"])
X = df_cleaned.drop(columns=["dBZ"])
y = df_cleaned["dBZ"]

In [ ]:
# ฟังก์ชันวัด MAE RMSE
def evaluate_model(X_imputed, y):
    X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse) # Calculate RMSE from MSE
    r2 = model.score(X_test_scaled, y_test) # Calculate R2 score
    return mae, rmse, r2

In [ ]:
# Evaluate the model using mean imputed data
mae, rmse, r2 = evaluate_model(X, y)

# --- แสดงผลลัพธ์ ---
print(f"MAE = {mae:.2f}, RMSE = {rmse:.2f}, R2 = {r2:.2f}")

MAE = 16.02, RMSE = 18.73, R2 = 0.03


In [ ]:
# 4) บันทึกเป็น Excel
output_path = "data_filled1.xlsx"
df.to_excel(output_path, index=False)

print(f"\n✅ บันทึกไฟล์เรียบร้อย: {output_path}")


✅ บันทึกไฟล์เรียบร้อย: data_filled1.xlsx
